In [1]:
import os
import sys

In [2]:
def iterate_over_files_in_directory(directory):
    for file_name in os.listdir(directory):
        if file_name.endswith(".csv"):  # You can adjust the file extension if needed
            file_path = os.path.join(directory, file_name)
            yield file_path

In [3]:
import os
import csv


In [4]:
import json
import warnings
import numpy as np
import csv
csv.field_size_limit(100000000)
        

131072

In [5]:
path = os.path.abspath('')+'/train'
y = []
X2 = []
LABELS = {"enterprise": 0, "cable": 1, "fiber": 2, "hotspot": 3, "eduroam": 4}

In [6]:
from collections import defaultdict
TCP_FLAG_VOCAB = ["SYN", "ACK", "FIN", "PSH", "RST", "URG", "ECE", "CWR"]
FLAG_TO_IDX = {f: i for i, f in enumerate(TCP_FLAG_VOCAB)}
K_FLAGS = len(TCP_FLAG_VOCAB)

def l2_normalize(X):
    X = X.astype(np.float32, copy=False)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms

def flags_to_multihot(flag_str, vocab=FLAG_TO_IDX):
    vec = np.zeros(len(vocab), dtype=np.float32)
    if not flag_str or flag_str == "NONE":
        return vec
    parts = flag_str.upper().replace(" ", "").split("+")
    for p in parts:
        if p in vocab:
            vec[vocab[p]] = 1.0
    return vec
def _parse_semicolon_list(s, cast=float):
    """Parse a semicolon-separated string into a list with type cast.
    Empty cells or '' return [].
    """
    if s is None:
        return []
    s = s.strip()
    if not s:
        return []
    parts = s.split(";")
    out = []
    for p in parts:
        p = p.strip()
        if p == "":
            # treat empty entries as missing; skip
            continue
        try:
            out.append(cast(p))
        except Exception:
            # if cast fails, skip that element
            continue
    return out

def traffic_csv_converter2(file_path, access_network, target_len=20, add_pkt_count=True):
    PAD_VAL = 0.0

    with open(file_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        return

    X_list, y_list = [], []

    for row in rows:
        # --- dst_port parsing ---
        try:
            dst_port = int(row.get("dst_ip") and row.get("dst_port") or row["dst_port"])
        except Exception:
            continue

        # skip DNS
        if dst_port == 53:
            continue

        # --- timestamps (seconds) ---
        ts_sec = _parse_semicolon_list(row.get("timestamps"), cast=float)
        if not ts_sec:
            continue

        ts_sec = np.array(ts_sec, dtype=np.float64)

        # remove flows with only 1 packet (or 0)
        if ts_sec.size <= 1:
            continue

        # --- inter-arrival times ---
        # timestamps in seconds -> convert to ms
        ts_ms = ts_sec * 1e3

        # i1 = 0, i_k = t_k - t_{k-1} (ms)
        iats_ms = np.diff(ts_ms, prepend=ts_ms[0])
        iats_ms[0] = 0.0

        # preprocess: clip [0,1000] ms and scale by 0.1
        iats = np.clip(iats_ms, 0.0, 1000.0) * 0.1

        # --- sizes ---
        sizes_list = _parse_semicolon_list(row.get("ip_sizes"), cast=float)
        if len(sizes_list) == 0:
            sizes_raw = np.zeros_like(ts_sec)
        else:
            sizes_raw = np.array(sizes_list, dtype=np.float64)
            if len(sizes_raw) == len(ts_sec):
                sizes_raw = sizes_raw
            else:
                sizes_raw = sizes_raw[:len(ts_sec)]
                if len(sizes_raw) < len(ts_sec):
                    sizes_raw = np.pad(
                        sizes_raw, (0, len(ts_sec) - len(sizes_raw)), constant_values=0.0
                    )

        # sizes are magnitudes, clipped to [0,1500]
        sizes = np.clip(np.abs(sizes_raw), 0.0, 1500.0)

        # --- truncate/pad to target_len (zero pad) ---
        L = min(len(ts_sec), target_len)

        feat_sz = sizes[:L]
        feat_iat = iats[:L]

        if L < target_len:
            pad = target_len - L
            feat_sz = np.pad(feat_sz, (0, pad), constant_values=PAD_VAL)
            feat_iat = np.pad(feat_iat, (0, pad), constant_values=PAD_VAL)

        # Concatenate as (s1..s_T, i1..i_T)  => length 2*target_len
        sample = np.concatenate([feat_sz, feat_iat]).astype(np.float32)

        if add_pkt_count:
            sample = np.concatenate([sample, np.array([np.float32(L)], dtype=np.float32)])

        X_list.append(sample)
        y_list.append(LABELS[access_network])

    if not X_list:
        return

    X2.extend(X_list)
    y.extend(y_list)


In [7]:
ethernet_path = path + "/enterprise"
for file_path in iterate_over_files_in_directory(ethernet_path):
        traffic_csv_converter2(file_path,"enterprise")
modem_path = path + "/cable"
for file_path in iterate_over_files_in_directory(modem_path):
        traffic_csv_converter2(file_path,"cable")
fiber_path = path + "/fiber"
for file_path in iterate_over_files_in_directory(fiber_path):
        traffic_csv_converter2(file_path,"fiber")
hotspot_path = path + "/hotspot"
for file_path in iterate_over_files_in_directory(hotspot_path):
        traffic_csv_converter2(file_path,"hotspot")

In [8]:
X2=np.array(X2)
y=np.array(y)

In [9]:
X2.shape

(139915, 41)

In [10]:
X_test = []
y_test = []

In [11]:
def test_traffic_csv_converter(file_path, access_network, target_len=20,add_pkt_count=True):
    PAD_VAL = 0.0

    with open(file_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        return

    X_list, y_list = [], []

    for row in rows:
        # --- dst_port parsing ---
        try:
            dst_port = int(row.get("dst_ip") and row.get("dst_port") or row["dst_port"])
        except Exception:
            continue

        # skip DNS
        if dst_port == 53:
            continue

        # --- timestamps (seconds) ---
        ts_sec = _parse_semicolon_list(row.get("timestamps"), cast=float)
        if not ts_sec:
            continue

        ts_sec = np.array(ts_sec, dtype=np.float64)

        # remove flows with only 1 packet (or 0)
        if ts_sec.size <= 1:
            continue

        # --- inter-arrival times ---
        # timestamps in seconds -> convert to ms
        ts_ms = ts_sec * 1e3

        # i1 = 0, i_k = t_k - t_{k-1} (ms)
        iats_ms = np.diff(ts_ms, prepend=ts_ms[0])
        iats_ms[0] = 0.0

        # preprocess: clip [0,1000] ms and scale by 0.1
        iats = np.clip(iats_ms, 0.0, 1000.0) * 0.1

        # --- sizes ---
        sizes_list = _parse_semicolon_list(row.get("ip_sizes"), cast=float)
        if len(sizes_list) == 0:
            sizes_raw = np.zeros_like(ts_sec)
        else:
            sizes_raw = np.array(sizes_list, dtype=np.float64)
            if len(sizes_raw) == len(ts_sec):
                sizes_raw = sizes_raw
            else:
                sizes_raw = sizes_raw[:len(ts_sec)]
                if len(sizes_raw) < len(ts_sec):
                    sizes_raw = np.pad(
                        sizes_raw, (0, len(ts_sec) - len(sizes_raw)), constant_values=0.0
                    )

        # sizes are magnitudes, clipped to [0,1500]
        sizes = np.clip(np.abs(sizes_raw), 0.0, 1500.0)

        # --- truncate/pad to target_len (zero pad) ---
        L = min(len(ts_sec), target_len)

        feat_sz = sizes[:L]
        feat_iat = iats[:L]

        if L < target_len:
            pad = target_len - L
            feat_sz = np.pad(feat_sz, (0, pad), constant_values=PAD_VAL)
            feat_iat = np.pad(feat_iat, (0, pad), constant_values=PAD_VAL)

        # Concatenate as (s1..s_T, i1..i_T)  => length 2*target_len
        sample = np.concatenate([feat_sz, feat_iat]).astype(np.float32)

        if add_pkt_count:
            sample = np.concatenate([sample, np.array([np.float32(L)], dtype=np.float32)])

        X_list.append(sample)
        y_list.append(LABELS[access_network])

    if not X_list:
        return

    X_test.extend(X_list)
    y_test.extend(y_list)

In [12]:
path = os.path.abspath('')+'/test'
ethernet_path = path + "/enterprise"
for file_path in iterate_over_files_in_directory(ethernet_path):
        test_traffic_csv_converter(file_path,"enterprise")
modem_path = path + "/cable"
for file_path in iterate_over_files_in_directory(modem_path):
        test_traffic_csv_converter(file_path,"cable")
fiber_path = path + "/fiber"
for file_path in iterate_over_files_in_directory(fiber_path):
        test_traffic_csv_converter(file_path,"fiber")
hotspot_path = path + "/hotspot"
for file_path in iterate_over_files_in_directory(hotspot_path):
        test_traffic_csv_converter(file_path,"hotspot")

In [13]:
X_test = np.array(X_test)
y_test = np.array(y_test)

In [19]:
import os
import time
import random
import numpy as np

import faiss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)
# import torch
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    # torch.manual_seed(seed)
    # torch.cuda.manual_seed_all(seed)

def ensure_2d(X: np.ndarray) -> np.ndarray:
    """
    FAISS expects (N, d). If you have (N, L, D), flatten to (N, L*D).
    """
    X = np.asarray(X)
    if X.ndim == 2:
        return X
    if X.ndim == 3:
        return X.reshape(X.shape[0], -1)
    raise ValueError(f"Expected X to be 2D or 3D, got shape {X.shape}")

def _faiss_sync(index):
    """
    Best-effort synchronization for accurate GPU timings.
    - Works if using a GPU index and faiss has cuda_synchronize.
    - Otherwise no-op (CPU).
    """
    try:
        # This exists in many FAISS GPU builds
        if hasattr(faiss, "cuda_synchronize"):
            faiss.cuda_synchronize()
            return
    except Exception:
        pass

    # Fallback: if you also have torch installed, this often syncs the device.
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    except Exception:
        pass

def knn_predict_majority(index, X_query, y_train_labels, k: int):
    """
    Majority vote among k nearest neighbors returned by FAISS.
    """
    Xq = np.ascontiguousarray(np.asarray(X_query, dtype=np.float32))
    _, I = index.search(Xq, k)  # I: (N, k) indices into the added vectors

    y_train_labels = np.asarray(y_train_labels)
    preds = np.empty(I.shape[0], dtype=np.int64)

    for i in range(I.shape[0]):
        labs = y_train_labels[I[i]]
        vals, counts = np.unique(labs, return_counts=True)
        preds[i] = vals[np.argmax(counts)]
    return preds

def build_faiss_index(X_train_2d_float32, metric: str = "l2", use_gpu: bool = True, gpu_id: int = 0):
    """
    Builds an IndexFlat (exact kNN) and moves to GPU if requested & available.
    metric: "l1" or "l2"
    """
    Xtr = np.ascontiguousarray(X_train_2d_float32, dtype=np.float32)
    d = Xtr.shape[1]

    metric = metric.lower()
    if metric == "l1":
        cpu_index = faiss.IndexFlat(d, faiss.METRIC_L1)
    elif metric == "l2":
        cpu_index = faiss.IndexFlatL2(d)
    else:
        raise ValueError("metric must be 'l1' or 'l2'")

    # Try to use GPU if requested and FAISS GPU is present
    if use_gpu:
        try:
            if hasattr(faiss, "get_num_gpus") and faiss.get_num_gpus() > 0:
                res = faiss.StandardGpuResources()
                gpu_index = faiss.index_cpu_to_gpu(res, gpu_id, cpu_index)
                return gpu_index
        except Exception:
            # Fall back to CPU below
            pass

    return cpu_index

def run_knn_faiss_one_seed(
    seed: int,
    X, y,
    X_test, y_test,
    *,
    k: int = 1,
    test_size: float = 0.2,
    use_standard_scaler: bool = True,
    metric: str = "l2",       # "l1" or "l2"
    use_gpu: bool = True,
    gpu_id: int = 0,
):
    set_all_seeds(seed)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=42,
        shuffle=True
    )

    # ---- ensure 2D for FAISS ----
    X_train = ensure_2d(X_train)
    X_val   = ensure_2d(X_val)
    X_test2 = ensure_2d(X_test)

    # -----------------------------
    # "Training" time = scaler fit/transform + index build + add
    # -----------------------------
    train_t0 = time.perf_counter()

    scaler_fit_time_s = 0.0
    scaler_transform_time_s = 0.0
    scaler = None

    if use_standard_scaler:
        t0 = time.perf_counter()
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        scaler_fit_time_s = time.perf_counter() - t0

        t0 = time.perf_counter()
        X_val   = scaler.transform(X_val)
        X_test2 = scaler.transform(X_test2)
        scaler_transform_time_s = time.perf_counter() - t0

    # ---- build index (CPU or GPU) ----
    Xtr_f32 = np.ascontiguousarray(X_train.astype(np.float32))

    t0 = time.perf_counter()
    index = build_faiss_index(Xtr_f32, metric=metric, use_gpu=use_gpu, gpu_id=gpu_id)
    index_create_time_s = time.perf_counter() - t0

    _faiss_sync(index)
    t0 = time.perf_counter()
    index.add(Xtr_f32)
    _faiss_sync(index)
    index_add_time_s = time.perf_counter() - t0

    train_time_s = time.perf_counter() - train_t0

    # -----------------------------
    # VAL inference (timed)
    # -----------------------------
    Xv_f32 = np.ascontiguousarray(X_val.astype(np.float32))
    _faiss_sync(index)
    t0 = time.perf_counter()
    y_val_pred = knn_predict_majority(index, Xv_f32, y_train, k=k)
    _faiss_sync(index)
    val_infer_time_s = time.perf_counter() - t0

    val_metrics = {
        "acc": float(accuracy_score(y_val, y_val_pred)),
        "macro_f1": float(f1_score(y_val, y_val_pred, average="macro")),
        "weighted_f1": float(f1_score(y_val, y_val_pred, average="weighted")),
        "macro_precision": float(precision_score(y_val, y_val_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_val, y_val_pred, average="macro", zero_division=0)),
        "cm": confusion_matrix(y_val, y_val_pred),
        "infer_time_s": float(val_infer_time_s),
        "infer_time_per_sample_ms": float(1000.0 * val_infer_time_s / max(1, len(y_val))),
    }

    Xt_f32 = np.ascontiguousarray(X_test2.astype(np.float32))
    _faiss_sync(index)
    t0 = time.perf_counter()
    y_test_pred = knn_predict_majority(index, Xt_f32, y_train, k=k)
    _faiss_sync(index)
    test_infer_time_s = time.perf_counter() - t0

    test_metrics = {
        "acc": float(accuracy_score(y_test, y_test_pred)),
        "macro_f1": float(f1_score(y_test, y_test_pred, average="macro")),
        "weighted_f1": float(f1_score(y_test, y_test_pred, average="weighted")),
        "macro_precision": float(precision_score(y_test, y_test_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_test, y_test_pred, average="macro", zero_division=0)),
        "cm": confusion_matrix(y_test, y_test_pred),
        "infer_time_s": float(test_infer_time_s),
        "infer_time_per_sample_ms": float(1000.0 * test_infer_time_s / max(1, len(y_test))),
    }

    return {
        "seed": int(seed),
        "train_time_s": float(train_time_s),
        "train_time_breakdown_s": {
            "scaler_fit": float(scaler_fit_time_s),
            "scaler_transform_val_test": float(scaler_transform_time_s),
            "index_create": float(index_create_time_s),
            "index_add": float(index_add_time_s),
        },
        "val": val_metrics,
        "test": test_metrics,
    }


def summarize(results, split="test"):
    keys = ["acc", "macro_f1", "weighted_f1", "macro_precision", "macro_recall"]

    print(f"\n=== {split.upper()} per run ===")
    for r in results:
        m = r[split]
        print(
            f"seed={r['seed']} | "
            f"acc={m['acc']:.4f} | "
            f"macro_f1={m['macro_f1']:.4f} | "
            f"weighted_f1={m['weighted_f1']:.4f} | "
            f"train_s={r['train_time_s']:.3f} | "
            f"{split}_infer_s={m['infer_time_s']:.6f} | "
            f"{split}_ms/sample={m['infer_time_per_sample_ms']:.6f}"
        )

    print(f"\n=== {split.upper()} Mean ± Std over {len(results)} runs ===")
    for k in keys:
        vals = np.array([r[split][k] for r in results], dtype=np.float64)
        if len(vals) > 1:
            print(f"{k:16s}: {vals.mean():.4f} ± {vals.std(ddof=1):.4f}")
        else:
            print(f"{k:16s}: {vals.mean():.4f}")

    train_vals = np.array([r["train_time_s"] for r in results], dtype=np.float64)
    infer_vals = np.array([r[split]["infer_time_s"] for r in results], dtype=np.float64)
    infer_msps = np.array([r[split]["infer_time_per_sample_ms"] for r in results], dtype=np.float64)

    print(f"\nTiming (Mean ± Std)")
    if len(train_vals) > 1:
        print(f"{'train_time_s':16s}: {train_vals.mean():.6f} ± {train_vals.std(ddof=1):.6f}")
        print(f"{'infer_time_s':16s}: {infer_vals.mean():.6f} ± {infer_vals.std(ddof=1):.6f}")
        print(f"{'infer_ms/sample':16s}: {infer_msps.mean():.6f} ± {infer_msps.std(ddof=1):.6f}")
    else:
        print(f"{'train_time_s':16s}: {train_vals.mean():.6f}")
        print(f"{'infer_time_s':16s}: {infer_vals.mean():.6f}")
        print(f"{'infer_ms/sample':16s}: {infer_msps.mean():.6f}")

    cms = np.stack([r[split]["cm"] for r in results], axis=0)
    print(f"\n{split.upper()} confusion matrix mean:\n", cms.mean(axis=0))
    if cms.shape[0] > 1:
        print(f"\n{split.upper()} confusion matrix std:\n", cms.std(axis=0, ddof=1))


ModuleNotFoundError: No module named 'torch'

In [20]:
seeds = [0, 1, 2, 3, 4]

results = [
    run_knn_faiss_one_seed(
        seed=s,
        X=X2, y=y,              # full training pool
        X_test=X_test, y_test=y_test,
        k=1,
        test_size=0.2,
        use_standard_scaler=True,
        metric="l1",            # matches your METRIC_L1
        gpu_id=0
    )
    for s in seeds
]

summarize(results, split="val")
summarize(results, split="test")


=== VAL per run ===
seed=0 | acc=0.6321 | macro_f1=0.6318 | weighted_f1=0.6323 | train_s=0.211 | val_infer_s=0.759252 | val_ms/sample=0.027133
seed=1 | acc=0.6321 | macro_f1=0.6318 | weighted_f1=0.6323 | train_s=0.209 | val_infer_s=0.674417 | val_ms/sample=0.024101
seed=2 | acc=0.6321 | macro_f1=0.6318 | weighted_f1=0.6323 | train_s=0.222 | val_infer_s=0.675804 | val_ms/sample=0.024151
seed=3 | acc=0.6321 | macro_f1=0.6318 | weighted_f1=0.6323 | train_s=0.209 | val_infer_s=0.676032 | val_ms/sample=0.024159
seed=4 | acc=0.6321 | macro_f1=0.6318 | weighted_f1=0.6323 | train_s=0.210 | val_infer_s=0.674807 | val_ms/sample=0.024115

=== VAL Mean ± Std over 5 runs ===
acc             : 0.6321 ± 0.0000
macro_f1        : 0.6318 ± 0.0000
weighted_f1     : 0.6323 ± 0.0000
macro_precision : 0.6326 ± 0.0000
macro_recall    : 0.6318 ± 0.0000

Timing (Mean ± Std)
train_time_s    : 0.212034 ± 0.005483
infer_time_s    : 0.692063 ± 0.037566
infer_ms/sample : 0.024732 ± 0.001342

VAL confusion matrix m